# ORPO — Odds Ratio Preference Optimization
### Direct / Reward-Free Alignment  ·  Colab T4 (16 GB) ready

> **DPO/IPO/KTO all assume SFT is already done** and all need a frozen reference `π_ref`.
> **ORPO needs neither.** It folds alignment *into* the supervised fine-tuning loss and drops the reference model entirely — one run, from a **base** model, no reference.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)
- **ORPO (Odds Ratio Preference Optimization)** (*Hong et al., "ORPO: Monolithic Preference Optimization without Reference Model"*) is a **single-stage, reference-free** alignment loss that trains a **base** (pretrained) model into an aligned model in **one pass**.
- It augments the standard **SFT negative-log-likelihood (NLL)** on the chosen response with a mild **odds-ratio penalty** that discourages the rejected response — so the *same* gradient step does **supervised learning and preference alignment together**.
- The contrast uses the **odds** of a sequence, not its probability ratio (DPO's choice):
  $$\text{odds}_\theta(y\mid x) = \frac{P_\theta(y\mid x)}{1 - P_\theta(y\mid x)}$$
  The odds ratio is a **gentler** contrast than the log-prob ratio, so it separates chosen from rejected **without** destabilizing the NLL term or needing a reference model to bound divergence.

### One-sentence definition of the mechanics
> **ORPO minimizes the SFT NLL of the chosen response plus a λ-weighted log-odds-ratio penalty that softly suppresses the rejected response, achieving supervised fine-tuning and preference alignment in a single reference-free training run from a base model.**

The loss:
$$\mathcal{L}_{\text{ORPO}} = \underbrace{\mathcal{L}_{\text{NLL}}(y_w\mid x)}_{\text{SFT: learn the chosen response}} \;+\; \lambda\cdot\underbrace{\Big(-\log \sigma\big(\log\tfrac{\text{odds}_\theta(y_w|x)}{\text{odds}_\theta(y_l|x)}\big)\Big)}_{\text{alignment: softly reject the loser}}$$

### The exact engineering problem it solves
- **The multi-stage pipeline tax.** The standard recipe is **two runs**: (1) SFT on demonstrations, then (2) DPO/IPO/KTO on preferences — each with its own data prep, checkpoints, hyperparameters, and a **frozen reference model** to hold in memory.
- **ORPO collapses it to one.** No separate SFT checkpoint, no reference `π_ref`, no second training job. The NLL term *is* the SFT; the odds-ratio term *is* the alignment. **Fewer moving parts, half the pipeline, less VRAM (no reference forward).**
- **Why it can drop the reference:** the NLL on the chosen response keeps the model anchored to fluent, plausible language, so there is no need for a KL-to-reference term to prevent degeneration — the supervised signal is the anchor.

---

### The Human Element — Hugging Face datasets for ORPO

| HF path | What it is | Why it's shaped this way for ORPO |
|---|---|---|
| **`mlabonne/orpo-dpo-mix-40k`** | The **canonical ORPO training mix** (~40k) — a curated blend (Capybara, UltraFeedback, distilled sets) in `(prompt, chosen, rejected)` conversational form. Purpose-built for the popular ORPO fine-tuning recipe. | ORPO's NLL trains **directly on the `chosen` text**, so `chosen` must be **high-quality demonstration data**, not just "better than rejected." This mix is filtered for exactly that — it doubles as SFT data *and* preference data. Used in Section 3. |
| **`HuggingFaceH4/ultrafeedback_binarized`** | GPT-4-scored `chosen`/`rejected` (split `train_prefs`). | High-quality `chosen` responses → good SFT signal for the NLL term; pairwise structure → the odds-ratio contrast. The same corpus you can A/B against DPO/IPO. |
| **`Anthropic/hh-rlhf`** | Human `(chosen, rejected)` pairs. | Human-authored `chosen` gives ORPO a human demonstration to imitate (NLL) while the pair drives the odds-ratio penalty. |

**Why ORPO still needs pairs (unlike KTO):** the loss has two jobs — **NLL needs the `chosen`** completion to imitate, and the **odds-ratio term needs the `rejected`** completion to contrast against. Both come from the *same* prompt, so ORPO consumes the standard `(prompt, chosen, rejected)` triple — but note the **chosen quality matters more** here than in DPO, because ORPO literally trains to reproduce it.

> This notebook trains on **`mlabonne/orpo-dpo-mix-40k`**, starting from a **base** (non-Instruct) model (Section 3).

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the mathematical reason ORPO works
- **Why the odds ratio, not the probability ratio?** DPO contrasts `log(π_θ/π_ref)` — a strong signal that can push the rejected probability arbitrarily low, which is why DPO **needs a reference** to keep the policy from collapsing. ORPO's **log-odds-ratio** `log(odds(y_w)/odds(y_l))` is a **weaker, saturating** contrast: it nudges the model to prefer chosen over rejected **without** driving rejected probability to zero, so it composes cleanly with the NLL term.
- **Why no reference model is needed.** In DPO/IPO/KTO the reference is what stops the policy from degenerating away from fluent language. In ORPO that job is done by the **NLL term itself** — the model is always being pulled toward reproducing a real, fluent `chosen` response. The odds-ratio penalty only adds a *relative* preference on top. **Anchor = supervised signal, not a reference model.**
- **Why one stage works.** SFT and preference optimization normally conflict if run together (alignment can undo SFT's fluency). ORPO's gentle odds-ratio term is *designed* to be co-trainable with NLL — the paper shows the log-odds-ratio yields a sensible gradient that **raises `P(y_w)` while mildly lowering `P(y_l)`** in the same step.

#### VRAM & Compute Impact
- **No reference model — at all.** Unlike DPO/IPO/KTO (which keep a reference, even if via adapter-disabling), ORPO computes everything from the single policy. **One model, and no reference forward pass.**
- **Two policy forwards per step** (chosen + rejected) for the odds-ratio contrast; the NLL reuses the chosen forward. Comparable per-step activation to DPO's policy side, **minus all reference bookkeeping**.
- **Half the pipeline.** One training run instead of SFT-then-DPO → **one dataset prep, one set of checkpoints, one hyperparameter search, less total wall-clock and infra.**
- **Capacity note:** because ORPO is *also* doing SFT (learning instruction-following from a **base** model), it benefits from **more capacity** than a pure preference tweak — hence higher LoRA rank / more epochs / more data below.

#### Pros & Cons

**Pros**
- **Simplest pipeline of all** — single stage, **no SFT checkpoint, no reference model, no reward/value model**.
- **Lowest reference-free VRAM** among these methods (no `π_ref` to compute or store).
- **Strong quality** — in the paper ORPO matched or beat the two-stage SFT→DPO recipe.
- **Starts from a base model** — skip acquiring/producing an SFT checkpoint entirely.

**Cons**
- **Still needs paired data** (chosen + rejected) — unlike KTO's unpaired signals.
- **Chosen quality is critical** — the NLL trains to reproduce `chosen`; garbage-in shows up directly in generations.
- **Needs more data/epochs & careful LR** than a pure preference pass, because it is *also* doing SFT from scratch.
- **`λ` (beta) is a balance knob** — too high destabilizes the NLL (fluency suffers), too low = weak alignment.
- **No explicit KL bound** — controllability rests entirely on the NLL anchor; less tunable divergence than reference-based methods.

#### Metrics to watch (TRL `ORPOTrainer` logs ORPO-specific keys)
- **`nll_loss`** — the SFT component; should **decrease** (the model is learning to produce `chosen`). **ORPO-distinctive.**
- **`log_odds_ratio` / `log_odds_chosen`** — the alignment component; the odds gap should **grow** then stabilize. **ORPO-distinctive.**
- **`rewards/chosen`, `rewards/rejected`, `rewards/margins`, `rewards/accuracies`** — the usual preference-separation signals; accuracy should climb.
- **`loss`** — the combined NLL + λ·OR objective.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit) + TRL `ORPOTrainer`, starting from a BASE model, with NO reference model.**

> ⚙️ **No reference, no SFT stage:** ORPO uses **`ORPOTrainer` / `ORPOConfig`** — there is **no `ref_model` argument** because ORPO is reference-free, and we start from **`Qwen/Qwen2.5-0.5B`** (the *base*, not `-Instruct`) because ORPO's NLL term *is* the SFT.

> ⚙️ **`beta` = the paper's `λ`:** it weights the odds-ratio penalty against the NLL term.

**Executable pipeline:**

| Step | What | Notes |
|---|---|---|
| 1 | 4-bit **base** `Qwen/Qwen2.5-0.5B` + LoRA + tokenizer | one policy, **no reference** |
| 2 | `mlabonne/orpo-dpo-mix-40k` → `(prompt, chosen, rejected)` | chosen doubles as SFT data |
| 3 | `ORPOConfig(beta=λ, …)` | NLL + odds-ratio in one loss |
| 4 | `ORPOTrainer.train()` | SFT + alignment, single run |
| 5 | Save adapter · export · inference | ship it |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch, os, math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import ORPOTrainer, ORPOConfig  # reference-free; note there is NO ref_model

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# The Colab T4 is Turing (compute capability 7.5). bfloat16 TENSOR CORES only exist on
# Ampere (SM 8.0) and newer. Forcing bf16 here makes the bitsandbytes 4-bit dequant path
# return garbage, which becomes NaN logits -> NaN softmax -> "CUDA error: device-side
# assert triggered" inside torch.multinomial at generation time. Detect, don't assume.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8                                   # True on A100/L4/H100, False on T4
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

### Step 1 — Quantization, LoRA & Tokenizer (from a **base** model)

The key difference from DPO/IPO/KTO: we load the **base** checkpoint `Qwen/Qwen2.5-0.5B` (**not** `-Instruct`), because ORPO's NLL term does the SFT itself. We give LoRA **more capacity** (`r=32`) since it is learning instruction-following *and* alignment. There is **no reference model** to configure.

In [ ]:
# BASE (pretrained) model — ORPO turns it into an aligned model in one run.
# (Qwen/Qwen2.5-0.5B is the base; -Instruct would already be SFT-ed, defeating the point.)
base_model_id = "Qwen/Qwen2.5-0.5B"

# 4-bit NF4 base weights (only ONE model here — no reference).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Qwen2.5 BASE ships the ChatML chat template; if you swap in a base model that lacks one,
# set tokenizer.chat_template before mapping the dataset (ORPO must format prompts consistently).
assert tokenizer.chat_template is not None, "Base tokenizer has no chat_template — set one first."

# LoRA with HIGHER rank than the pure-preference notebooks: ORPO also does SFT from base,
# so it needs more capacity to learn instruction-following, not just a preference nudge.
peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

policy_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # mem-efficient attention (T4 has no FlashAttn-2)
)
policy_model.config.use_cache = False  # required with gradient checkpointing
print(policy_model.get_memory_footprint() / 1e9, "GB base (4-bit)  — and NO reference model")

### Step 2 — Dataset: `mlabonne/orpo-dpo-mix-40k`

The canonical ORPO mix, in `(prompt, chosen, rejected)` conversational form. Remember: **`chosen` is also the SFT target** (the NLL trains on it), so its quality matters. We flatten to the standard string triple (templated prompt + raw completion text).

In [ ]:
raw = load_dataset("mlabonne/orpo-dpo-mix-40k", split="train")
raw = raw.shuffle(seed=42).select(range(1000))  # subset so a T4 finishes in a reasonable time

def to_orpo_triple(ex):
    # chosen / rejected are conversational lists ending in the assistant turn and sharing
    # the same prompt turns. Everything but the final assistant message is the prompt.
    prompt_msgs = ex["chosen"][:-1]
    return {
        # add_generation_prompt=True => ends with the assistant header the model completes.
        "prompt":   tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True),
        "chosen":   ex["chosen"][-1]["content"],    # SFT target (NLL) AND odds-ratio winner
        "rejected": ex["rejected"][-1]["content"],  # odds-ratio loser
    }

# String prompt/chosen/rejected => TRL treats it as STANDARD format (no re-templating).
orpo_ds = raw.map(to_orpo_triple, remove_columns=raw.column_names)
print(orpo_ds)

In [ ]:
# Step 2.5 — Free leftover GPU memory (run before training)

import gc
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB  (expect several GB free before training)")

### Step 3 — `ORPOConfig` (NLL + odds-ratio) & `ORPOTrainer`

**`beta` is the paper's `λ`** — the weight on the odds-ratio penalty relative to the NLL. Because ORPO is *also* doing SFT from a base model, we use a **slightly higher LR** and **more epochs** than the pure-preference notebooks. There is **no `ref_model`** — ORPO is reference-free.

In [ ]:
"""
- Compute warmup steps dynamically

`warmup_steps` is a step count, not a ratio — derive it from the actual optimizer-step total
(`rows * epochs / (batch * grad_accum)`) so it scales automatically if you change the subset size,
epoch count, or batch settings above. **~10% warmup** is the standard preference-tuning default.
"""

per_device_train_batch_size = 1
gradient_accumulation_steps = 8
num_train_epochs = 3

# Optimizer steps = passes through the data / effective batch size (world_size=1 on a single T4).
total_optimizer_steps = math.ceil(len(orpo_ds) * num_train_epochs / (per_device_train_batch_size * gradient_accumulation_steps))
warmup_steps = max(1, math.ceil(total_optimizer_steps * 0.1))  # ~10% warmup
print(f"optimizer steps: {total_optimizer_steps}  ->  warmup_steps: {warmup_steps}")

In [ ]:
orpo_config = ORPOConfig(
    output_dir="./orpo_output",
    run_name="orpo-t4",

    # ---- The ORPO knob ----
    beta=0.1,   # lambda in the paper: weight of the odds-ratio penalty vs the NLL/SFT term.
                # Too high -> NLL destabilizes (fluency drops); too low -> weak alignment.

    # ---- Sequence budget (chosen AND rejected tokenized for the odds ratio) ----
    max_length=512,        # cap on prompt + completion
    max_prompt_length=256,  # prompt-only cap

    # ---- T4 16 GB hardening ----
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,  # effective batch = 8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16, fp16=not USE_BF16,  # match the GPU: fp16 on T4, bf16 on Ampere+
    optim="paged_adamw_8bit",

    # ---- Optimization schedule (ORPO is ALSO doing SFT -> a bit hotter/longer) ----
    learning_rate=1e-5,          # slightly higher than pure DPO/IPO/KTO; push to 2e-5 if underfitting
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    num_train_epochs=num_train_epochs,  # more epochs: it must learn instruction-following from BASE
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# NO ref_model argument — ORPO is reference-free. One policy, one loss.
orpo_trainer = ORPOTrainer(
    model=policy_model,
    args=orpo_config,
    train_dataset=orpo_ds,
    processing_class=tokenizer,  
    peft_config=peft_config,
)

### Step 4 — Train (SFT + alignment, one run)

Watch **both** components move: **`nll_loss`** should fall (the model learns to produce `chosen` — the SFT job) while **`log_odds_chosen`** and **`rewards/margins`** grow (the alignment job) — happening in the *same* steps.

In [ ]:
orpo_trainer.train()

# Save the aligned LoRA adapter (a few MB, not GB).
orpo_trainer.save_model("./orpo_aligned_adapter")
tokenizer.save_pretrained("./orpo_aligned_adapter")
print("Saved -> ./orpo_aligned_adapter")

## Export — Download the Aligned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./orpo_aligned_adapter"
output_filename = "orpo_aligned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — Evaluate the ORPO-Aligned Policy

Reload the **base** `Qwen/Qwen2.5-0.5B` + the ORPO adapter and generate with the **chat template** ORPO trained on. (This is the payoff: a **base** model that now follows instructions — SFT and alignment both came from the single ORPO run.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Re-derive the dtype here so this cell works standalone after a restart.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

base_model_id = "Qwen/Qwen2.5-0.5B"   # the BASE model (ORPO did the SFT)
adapter_path = "./orpo_aligned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_path)  # attach the ORPO adapter
model.eval()

In [ ]:
# Same chat template the ORPO loop used (single user turn).
def generate_response(user_prompt, max_new_tokens=256, temperature=0.7):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Decode only the newly generated completion (slice off the prompt tokens).
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "How does ORPO align a model without a separate SFT stage or a reference model?",
    "My friend has been feeling really down lately. How can I support them?",
]

print("--- ORPO-Aligned Responses (from a BASE model) ---")
for i, p in enumerate(test_prompts, 1):
    print(f"\n[Prompt {i}]: {p}")
    print(f"[Response]: {generate_response(p).strip()}")
    print("-" * 60)